#### LangSmith 기본 예제


##### 1) 라이브러리 설치

In [ ]:
# uv add langsmith

##### 2) 환경 변수 설정 (Groq, LangSmith)

- `.env` 파일에 `GROQ_API_KEY`, `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT` 등록
- `load_dotenv()`가 `.env` 값을 환경 변수로 설정하므로 LangSmith가 별도 코드 없이 읽어서 사용

In [1]:
import os
from dotenv import load_dotenv
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

# 필수 환경 변수 확인 (값은 출력하지 않고 설정 여부만 확인)
REQUIRED_ENV = ["GROQ_API_KEY", "LANGSMITH_API_KEY", "LANGSMITH_PROJECT"]
missing = [name for name in REQUIRED_ENV if not os.getenv(name)]
if missing:
    raise RuntimeError(f".env 파일에 다음 환경 변수가 없습니다: {', '.join(missing)}")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# LANGSMITH_TRACING 이 'true' 여야 트레이스가 LangSmith 로 전송됨
print("LangSmith 트레이싱:", os.getenv("LANGSMITH_TRACING"))
print("LangSmith 프로젝트:", os.getenv("LANGSMITH_PROJECT"))

LangSmith 트레이싱: true
LangSmith 프로젝트: MyLangchainProject


##### LangSmith와 LangChain을 활용한 기본 로깅 예제

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langsmith import traceable

# LangSmith 설정(LANGSMITH_TRACING / API_KEY / PROJECT / ENDPOINT)은
# 위에서 load_dotenv() 로 환경 변수에 이미 로드되어 있으므로 다시 대입할 필요가 없음

# LLM 모델 설정 (Groq 사용: OpenAI 호환 API)
llm = ChatOpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",  # Groq API 엔드포인트
    model="openai/gpt-oss-120b",  
    temperature=0.7
)

# LangSmith로 실행 추적
@traceable(run_type="chain", name="Simple_Chain")
def ask_question(question: str):

    # 개별 메시지 템플릿 정의
    system_message = SystemMessagePromptTemplate.from_template(
        "당신은 유용한 AI 비서입니다."
    )
    user_message = HumanMessagePromptTemplate.from_template(
        "{question}"
    )

    chat_prompt = ChatPromptTemplate.from_messages([
        system_message,
        user_message,
    ])
    
    messages = chat_prompt.format_messages(question=question)
    response = llm.invoke(messages)

    return response.content

# 테스트 실행
question = "LangGraph와 LangChain의 차이점은 무엇인가요?"
answer = ask_question(question)

# 결과 출력
print("\n🔹 [AI 답변]:")
print(answer)



🔹 [AI 답변]:
## LangChain vs. LangGraph – 핵심 차이점 정리  

| 구분 | LangChain | LangGraph |
|------|-----------|-----------|
| **주된 목표** | LLM을 **시퀀스(Chain)** 로 연결해 간단·중간 규모의 워크플로우를 빠르게 구현 | LLM 기반 **그래프(Flow)** 로 복잡한 상태 전이, 조건 분기, 루프, 병렬 실행 등을 손쉽게 모델링 |
| **핵심 추상화** | `Chain`, `Agent`, `Tool`, `Memory` 등 **선형/계층형** 구조 | `Graph`, `Node`, `Edge`, `State` 등 **노드‑엣지** 기반 **상태 머신** |
| **제어 흐름** | 순차적 실행이 기본이며, `if‑else`·`while` 등은 Python 로 직접 구현 | 그래프 정의 시 **조건부 전이**, **반복**, **병렬** 등을 선언적으로 지정 |
| **상태 관리** | 메모리(`ConversationBufferMemory`, `ConversationSummaryMemory` 등) 를 체인에 붙여 사용 | **전역 상태 객체**(예: `StateDict`) 를 그래프 전체에 공유·읽고‑쓰기 가능, 상태 변화를 자동 추적 |
| **디버깅·관찰** | 로그와 콜백을 통해 각 체인 단계 확인 | 노드별 입력·출력·상태 변화를 시각화(플로우 차트)하고, 실행 히스토리를 자동 저장 |
| **재사용·구성** | 체인·에이전트를 재조합하지만, 복잡한 흐름을 표현하려면 중첩 체인·조건문이 필요 | **서브그래프**(sub‑graph) 로 모듈화가 자연스러워, 재귀 호출·재사용이 용이 |
| **배포·운영** | 기본적으로 Python 코드 + API 호출 형태 | 그래프 정의를 JSON/YAML 로 내보내어 **플러그인형** 배포가 가능(예: LangGraph Server) |
| **에코시스템** | 풍부한 **툴·프롬프트·메모리·체인** 템플릿 